# 03 - Team Feature Engineering

Goal: turn player-level stats into one ML-ready row per national team.

Why this step matters: a World Cup model does not play 3,552 individual players. It predicts teams. So we need to compress player stats into team-level signals like attack strength, chance creation, defensive activity, goalkeeper quality, squad depth, and data confidence.

Important modeling decision: we should not simply sum every player from a country. France, Spain, England, and Germany have hundreds of players in the dataset, while smaller countries have fewer. A raw sum would reward database coverage instead of football strength. So this notebook uses official parsed squad players when available, caps longlists to the strongest 26, and falls back to a probable top-26 squad only when a country's squad file is missing.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

SRC_DIR = Path('../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from team_feature_engineering import (
    PROCESSED_DIR,
    read_processed_csv,
    aggregate_team_squads,
    build_power_scores,
)

print('Notebook 03 imports loaded')

Notebook 03 imports loaded


## Step 1 - Load processed player data

This notebook starts from `player_ml_features.csv`, which was created by Notebook 02. That file already merged:

- 2025-26 current season data
- 2023-24 previous season xG/xAG/progressive stats
- 2014-2025 historical trends
- supplemental stars and estimated players

As a rule, we print columns before engineering features. This keeps us honest and prevents code from silently depending on columns that do not exist.

In [2]:
player_ml = read_processed_csv('player_ml_features.csv')

squad_coverage_path = PROCESSED_DIR / 'squad_nation_coverage.csv'
squad_coverage = pd.read_csv(squad_coverage_path) if squad_coverage_path.exists() else None

if squad_coverage is not None:
    print('\nSquad coverage columns:')
    print(list(squad_coverage.columns))
    display(squad_coverage.head())

player_ml_features.csv: 3552 rows x 72 columns
['player_key', 'player_name', 'nation_code', 'nation_name', 'weighted_goal_form', 'weighted_chance_form', 'weighted_minutes', 'cur_player', 'cur_position', 'cur_squad', 'cur_competition', 'cur_nation', 'cur_stat_source', 'cur_minutes', 'cur_nineties', 'cur_matches', 'cur_goals', 'cur_assists', 'cur_xg', 'cur_shots', 'cur_shots_on_target', 'cur_interceptions', 'cur_tackles_won', 'cur_save_pct', 'cur_goals_against', 'cur_clean_sheets', 'cur_goals_per90', 'cur_assists_per90', 'cur_xg_per90', 'cur_shots_per90', 'cur_shots_on_target_per90', 'cur_interceptions_per90', 'cur_tackles_won_per90', 'prev_player', 'prev_position', 'prev_squad', 'prev_competition', 'prev_nation', 'prev_minutes', 'prev_nineties', 'prev_matches', 'prev_goals', 'prev_assists', 'prev_xg', 'prev_npxg', 'prev_xag', 'prev_progressive_passes', 'prev_progressive_carries', 'prev_progressive_receptions', 'prev_goals_per90', 'prev_assists_per90', 'prev_xg_per90', 'prev_npxg_per90',

,nation_code,nation_name,squad_players,covered_players,supplemental_players,coverage_pct
0,CIV,Cote d'Ivoire,26,24,9,0.923077
1,CUW,Curacao,26,25,25,0.961538
2,HAI,Haiti,26,25,21,0.961538
3,EGY,Egypt,23,23,21,1.000000
4,GER,Germany,25,25,1,1.000000


## Step 2 - Select each country's squad representation

A national team squad has about 26 players. Our raw player file may contain 500+ Spanish players or 400+ French players, but Spain cannot take 500 players to the World Cup.

So the code gives each player a `selection_score` based on:

- `weighted_minutes`: players actually playing club football are more likely to matter
- `weighted_goal_form`: recent and historical scoring form
- `weighted_chance_form`: xG/xAG/chance contribution where available

The selection rule is:

1. If the country exists in `wc_squads_parsed.csv`, use those official/parsed squad players.
2. If that parsed list has more than 26 matched players, treat it as a longlist and keep the strongest 26 from that list.
3. If the country does not have a parsed squad yet, fall back to the strongest probable top 26 from the player database.

This is better than pure top-26 guessing because official squad information wins whenever we have it.

In [3]:
squad_player_coverage_path = PROCESSED_DIR / 'squad_player_coverage.csv'
squad_player_coverage = pd.read_csv(squad_player_coverage_path) if squad_player_coverage_path.exists() else None

probable_squads = aggregate_team_squads(player_ml, squad_player_coverage, squad_size=26)

print('probable_squads shape:', probable_squads.shape)
print('columns:')
print(list(probable_squads.columns))

display(probable_squads.head(10))

probable_squads shape: (45, 16)
columns:
['nation_code', 'nation_name', 'selection_method', 'squad_players_used', 'estimated_players_used', 'verified_supplemental_used', 'avg_selection_score', 'top6_goal_form_sum', 'top6_goal_form_mean', 'top10_chance_form_sum', 'top10_chance_form_mean', 'attacker_goal_form_mean', 'midfielder_chance_form_mean', 'defender_actions_per90_mean', 'keeper_save_pct_best', 'squad_weighted_minutes_mean']


,nation_code,nation_name,selection_method,squad_players_used,estimated_players_used,verified_supplemental_used,avg_selection_score,top6_goal_form_sum,top6_goal_form_mean,top10_chance_form_sum,top10_chance_form_mean,attacker_goal_form_mean,midfielder_chance_form_mean,defender_actions_per90_mean,keeper_save_pct_best,squad_weighted_minutes_mean
0,ARG,Argentina,official_squad_capped_top26,26,4,1,79.975258,2.476049,0.412675,1.330950,0.133095,0.311979,0.062294,2.320387,0.0,1774.823077
1,AUS,Australia,probable_top26,5,0,0,45.727055,0.021959,0.004392,0.072566,0.014513,0.000000,0.023201,1.756757,69.9,1164.640000
2,AUT,Austria,official_squad,26,7,0,64.514304,1.822630,0.303772,1.005248,0.100525,0.259417,0.056519,2.032390,68.0,1454.446154
3,BEL,Belgium,official_squad,26,5,0,69.303453,3.324222,0.554037,1.007210,0.100721,0.545739,0.065909,1.960641,74.5,1644.876923
4,BIH,Bosnia and Herzegovina,official_squad,26,21,0,62.162162,1.632371,0.272062,0.680442,0.068044,0.307131,0.034935,2.123074,68.0,1265.957692
5,BRA,Brazil,official_squad,26,5,3,69.741857,2.589309,0.431552,1.073079,0.107308,0.323679,0.066974,1.608294,68.0,1640.315385
6,CAN,Canada,probable_top26,11,0,0,50.187474,1.312158,0.218693,0.382567,0.038257,0.113761,0.028248,1.697797,0.0,724.690909
7,CIV,Cote d'Ivoire,official_squad,24,8,1,62.658801,1.633878,0.272313,0.619109,0.061911,0.238669,0.042910,1.979193,68.0,1289.066667
8,COD,DR Congo,official_squad,26,14,0,54.115504,1.067248,0.177875,0.611928,0.061193,0.227088,0.027710,2.083987,88.2,1026.350000
9,COL,Colombia,official_squad,26,14,2,63.886137,1.648844,0.274807,1.094096,0.109410,0.179729,0.059938,1.734220,68.0,1349.246154


## Step 3 - Understand the engineered squad features

The important columns are:

- `top6_goal_form_sum`: attacking threat from the six strongest goal-contributors
- `top10_chance_form_sum`: chance creation from the ten strongest creative players
- `attacker_goal_form_mean`: average scoring form among probable attackers
- `midfielder_chance_form_mean`: average chance creation among probable midfielders
- `defender_actions_per90_mean`: tackles plus interceptions per 90 among probable defenders
- `keeper_save_pct_best`: best goalkeeper save percentage found in the probable squad
- `selection_method`: whether we used official squad, capped official longlist, or probable top-26 fallback
- `estimated_players_used`: how many selected players came from estimated supplemental rows

The estimated-player count is not used to say a team is bad. It is used to say our data is less certain.

In [4]:
cols_to_view = [
    'nation_code', 'nation_name', 'selection_method', 'squad_players_used',
    'top6_goal_form_sum', 'top10_chance_form_sum',
    'defender_actions_per90_mean', 'keeper_save_pct_best',
    'estimated_players_used', 'verified_supplemental_used'
]

display(
    probable_squads[cols_to_view]
    .sort_values('top6_goal_form_sum', ascending=False)
    .head(20)
)

,nation_code,nation_name,selection_method,squad_players_used,top6_goal_form_sum,top10_chance_form_sum,defender_actions_per90_mean,keeper_save_pct_best,estimated_players_used,verified_supplemental_used
18,FRA,France,official_squad,26,3.649274,1.276214,1.728280,77.6,1,0
3,BEL,Belgium,official_squad,26,3.324222,1.007210,1.960641,74.5,5,0
16,ENG,England,official_squad,26,3.281209,1.454975,2.081289,73.7,1,0
28,NED,Netherlands,probable_top26,26,3.035569,1.427656,1.863835,0.0,0,0
19,GER,Germany,official_squad,25,2.591882,1.338744,2.019880,72.8,1,0
5,BRA,Brazil,official_squad,26,2.589309,1.073079,1.608294,68.0,5,3
33,POR,Portugal,official_squad,26,2.521718,1.510938,1.764184,68.0,4,2
17,ESP,Spain,official_squad,26,2.518759,0.963443,2.317031,80.4,2,0
0,ARG,Argentina,official_squad_capped_top26,26,2.476049,1.330950,2.320387,0.0,4,1
29,NOR,Norway,official_squad,26,2.448155,1.013177,1.725148,83.3,9,0


## Step 4 - Convert raw features into comparable scores

The raw columns have different units. Goals per 90, save percentage, minutes, and progressive metrics are not directly comparable.

So the code converts each signal into a percentile score from 0 to 100. Example: if Argentina is in the 90th percentile for attack, it means Argentina's attacking signal is stronger than about 90% of teams in this dataset.

Then we build five football scores:

- `attack_score`: goals and attacking player form
- `creation_score`: chance creation and creative depth
- `defense_score`: defensive actions and squad minutes
- `keeper_score`: goalkeeper signal
- `depth_score`: overall quality of the top 26

Finally, `power_score` combines them with weights. The first version is intentionally simple and explainable. Later, the ML model can learn better weights from historical match outcomes.

In [5]:
final_team_features = build_power_scores(probable_squads, squad_coverage)

print('final_team_features shape:', final_team_features.shape)
print('columns:')
print(list(final_team_features.columns))

display(final_team_features.head(15))

final_team_features shape: (45, 26)
columns:
['nation_code', 'nation_name', 'selection_method', 'squad_players_used', 'estimated_players_used', 'verified_supplemental_used', 'avg_selection_score', 'top6_goal_form_sum', 'top6_goal_form_mean', 'top10_chance_form_sum', 'top10_chance_form_mean', 'attacker_goal_form_mean', 'midfielder_chance_form_mean', 'defender_actions_per90_mean', 'keeper_save_pct_best', 'squad_weighted_minutes_mean', 'attack_score', 'creation_score', 'defense_score', 'keeper_score', 'depth_score', 'data_confidence_score', 'coverage_pct', 'has_announced_squad_file', 'raw_power_score', 'power_score']


,nation_code,nation_name,selection_method,squad_players_used,estimated_players_used,verified_supplemental_used,avg_selection_score,top6_goal_form_sum,top6_goal_form_mean,top10_chance_form_sum,...,attack_score,creation_score,defense_score,keeper_score,depth_score,data_confidence_score,coverage_pct,has_announced_squad_file,raw_power_score,power_score
0,ENG,England,official_squad,26,1,0,74.863565,3.281209,0.546868,1.454975,...,95.000000,96.777778,78.444444,86.666667,88.888889,98.653846,1.0,True,90.853333,90.547577
1,GER,Germany,official_squad,25,1,0,76.807714,2.591882,0.431980,1.338744,...,92.444444,94.444444,71.555556,84.444444,93.333333,98.600000,1.0,True,88.313333,88.004237
2,FRA,France,official_squad,26,1,0,77.163462,3.649274,0.608212,1.276214,...,96.555556,88.888889,45.555556,93.333333,95.555556,98.653846,1.0,True,84.972222,84.686258
3,ESP,Spain,official_squad,26,2,0,75.382369,2.518759,0.419793,0.963443,...,81.888889,74.555556,91.555556,95.555556,91.111111,97.307692,1.0,True,84.357778,83.789985
4,ARG,Argentina,official_squad_capped_top26,26,4,1,79.975258,2.476049,0.412675,1.330950,...,85.222222,88.000000,91.333333,18.888889,97.777778,94.615385,1.0,True,80.312222,79.231096
5,SEN,Senegal,official_squad_capped_top26,26,4,3,69.021922,2.389483,0.398247,1.042152,...,82.333333,73.555556,84.000000,77.777778,80.000000,94.615385,1.0,True,79.658889,78.586558
6,NED,Netherlands,probable_top26,26,0,0,80.394036,3.035569,0.505928,1.427656,...,93.666667,95.777778,52.222222,18.888889,100.000000,100.000000,1.0,False,78.394444,78.394444
7,BEL,Belgium,official_squad,26,5,0,69.303453,3.324222,0.554037,1.007210,...,90.666667,77.222222,52.666667,88.888889,82.222222,93.269231,1.0,True,79.407778,78.071589
8,POR,Portugal,official_squad,26,4,2,70.827243,2.521718,0.420286,1.510938,...,88.222222,95.555556,46.888889,56.666667,86.666667,94.615385,1.0,True,78.673333,77.614269
9,BRA,Brazil,official_squad,26,5,3,69.741857,2.589309,0.431552,1.073079,...,85.111111,84.666667,28.666667,56.666667,84.444444,93.269231,1.0,True,71.360000,70.159231


## Step 5 - Inspect the first power ranking

This is not the final World Cup prediction yet. It is a squad-strength ranking.

The difference:

- Squad-strength ranking asks: who looks strongest on paper?
- Tournament prediction asks: who wins after group path, knockout matchups, randomness, and simulations?

This ranking is still useful because it becomes an input feature for the actual match model.

In [6]:
ranking_cols = [
    'nation_code', 'nation_name', 'power_score', 'raw_power_score',
    'attack_score', 'creation_score', 'defense_score', 'keeper_score',
    'depth_score', 'data_confidence_score', 'estimated_players_used',
    'has_announced_squad_file'
]

power_ranking = final_team_features[ranking_cols].sort_values('power_score', ascending=False)
display(power_ranking.head(25))

,nation_code,nation_name,power_score,raw_power_score,attack_score,creation_score,defense_score,keeper_score,depth_score,data_confidence_score,estimated_players_used,has_announced_squad_file
0,ENG,England,90.547577,90.853333,95.000000,96.777778,78.444444,86.666667,88.888889,98.653846,1,True
1,GER,Germany,88.004237,88.313333,92.444444,94.444444,71.555556,84.444444,93.333333,98.600000,1,True
2,FRA,France,84.686258,84.972222,96.555556,88.888889,45.555556,93.333333,95.555556,98.653846,1,True
3,ESP,Spain,83.789985,84.357778,81.888889,74.555556,91.555556,95.555556,91.111111,97.307692,2,True
4,ARG,Argentina,79.231096,80.312222,85.222222,88.000000,91.333333,18.888889,97.777778,94.615385,4,True
5,SEN,Senegal,78.586558,79.658889,82.333333,73.555556,84.000000,77.777778,80.000000,94.615385,4,True
6,NED,Netherlands,78.394444,78.394444,93.666667,95.777778,52.222222,18.888889,100.000000,100.000000,0,False
7,BEL,Belgium,78.071589,79.407778,90.666667,77.222222,52.666667,88.888889,82.222222,93.269231,5,True
8,POR,Portugal,77.614269,78.673333,88.222222,95.555556,46.888889,56.666667,86.666667,94.615385,4,True
9,BRA,Brazil,70.159231,71.360000,85.111111,84.666667,28.666667,56.666667,84.444444,93.269231,5,True


## Step 6 - Save ML-ready team features

We save two files:

- `probable_squad_features.csv`: raw engineered squad features
- `final_team_features.csv`: squad features plus normalized football scores

Notebook 04 will use `final_team_features.csv` for baseline modeling and eventually match prediction.

In [7]:
probable_path = PROCESSED_DIR / 'probable_squad_features.csv'
final_path = PROCESSED_DIR / 'final_team_features.csv'

probable_squads.to_csv(probable_path, index=False)
final_team_features.to_csv(final_path, index=False)

print('Saved:', probable_path)
print('Saved:', final_path)

Saved: C:\Users\sambi\OneDrive\Desktop\worldcup-predictor\data\processed\probable_squad_features.csv
Saved: C:\Users\sambi\OneDrive\Desktop\worldcup-predictor\data\processed\final_team_features.csv


## What we learned

The main ML lesson from this notebook is that feature engineering is not just writing formulas. It is deciding what real-world object each row represents.

Here, the row represents a national team, but the raw data represents players. So we had to choose a fair aggregation method. Using the top 26 probable players is a reasonable football-aware compromise.

The next notebook should move from static team strength to match prediction. For that we need historical match results, because the model needs examples of Team A vs Team B and the final result.